# 13. Opsional — CNN kecil untuk angka (PyTorch)

**Capaian:** Mendefinisikan convolution 2-D dan membandingkan dengan fitur klasik.

**Prasyarat:** Python dasar, NumPy, dan membaca array. **Estimasi:** 75–100 menit.

> Kerjakan dari atas ke bawah. Setiap grafik wajib disertai penafsiran, bukan hanya ditempel.

## Konsep dan model

**Modul opsional:** jalankan setelah `pip install torch` sesuai petunjuk resmi PyTorch untuk OS/CPU/GPU Anda. CNN mempelajari filter dari data. Jumlah epoch sedikit adalah contoh didaktik, bukan benchmark atau pencarian arsitektur optimal.

## Impor dan siapkan tensor

In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
torch.manual_seed(42)
d=load_digits(); X=(d.images.astype('float32')/16)[:,None,:,:]
y=d.target.astype('int64')
tr,te=train_test_split(np.arange(len(y)),test_size=0.25,stratify=y,random_state=42)
Xtr=torch.from_numpy(X[tr]);ytr=torch.from_numpy(y[tr]);Xte=torch.from_numpy(X[te]);yte=y[te]
print('Tensor batch (N,C,H,W):',Xtr.shape)

## Arsitektur dan pelatihan CPU sederhana

In [ ]:
torch.set_num_threads(min(torch.get_num_threads(),4))
net=nn.Sequential(nn.Conv2d(1,8,kernel_size=3,padding=1),nn.ReLU(),
                  nn.MaxPool2d(2),nn.Conv2d(8,16,kernel_size=3,padding=1),nn.ReLU(),
                  nn.Flatten(),nn.Linear(16*4*4,10))
optimizer=torch.optim.Adam(net.parameters(),lr=0.01)
loss_fn=nn.CrossEntropyLoss()
net.train()
for epoch in range(4):
    order=torch.randperm(len(Xtr))
    for chunk in order.split(128):
        optimizer.zero_grad(); logits=net(Xtr[chunk]);loss=loss_fn(logits,ytr[chunk]);
        loss.backward();optimizer.step()
    print('Epoch',epoch+1,'loss batch terakhir:',round(loss.item(),3))

## Evaluasi dengan test tersimpan

In [ ]:
net.eval()
with torch.no_grad():
    pred=net(Xte).argmax(dim=1).numpy()
print(classification_report(yte,pred,zero_division=0,digits=3))
fig,ax=plt.subplots(1,6,figsize=(10,2))
for a,i in zip(ax,range(6)):
    a.imshow(Xte[i,0],cmap='gray');a.set_title(f'{yte[i]} → {pred[i]}');a.axis('off')
plt.show()

## Diskusi, HOTS, dan latihan

1. Jelaskan perbedaan *kernel* pada convolution dan pooling.
2. Bandingkan macro-F1 CNN dengan notebook 09 tanpa menganggap satu uji sebagai simpulan universal.
3. Apa efek batch size, optimizer, dan augmentasi terhadap overfitting?
4. Mengapa analisis performa pada data luar distribusi tetap diperlukan?

**Laporan singkat:** rumusan masalah → rancangan eksperimen → hasil → interpretasi → keterbatasan → kesimpulan.